# 07 — Activation Patching — Contribution C2

Causal mechanistic analysis of **DeiT-Base** and **DINOv2-ViT-B/14**
on CRC histology (TUM vs NORM binary discrimination task).

## Protocol

```
Trained ViT
  → Define binary task: TUM (8) vs NORM (6)
  → Generate corrupt image pairs (null_baseline, patch_shuffle, color_jitter …)
  → Attribution patching scan  (fast, all layers × components)
  → Exact activation patching  (top components, denoising + noising)
  → Per-head attribution        (which attention heads matter?)
  → Shortcut detection          (does the model cheat on color / texture?)
  → IE heatmaps + circuit diagram
  → Save + upload to Drive
```

**Reference**: Heimersheim & Nanda 2024 — "How to use and interpret activation patching"
(arxiv.org/abs/2404.15255)

**Edit only** `USER CONFIG` to switch between models or tasks.

**Environment**: Kaggle GPU T4 / P100 — Phase 1 only.

In [ ]:
# 0. Kaggle setup
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru
!pip install -q PyDrive2
!pip install -q torchvision scikit-learn

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
# ============================================================
# USER CONFIG — edit this cell only
# ============================================================

# Primary targets: deit_base | dinov2
MODEL_NAME = "deit_base"

# Google Drive folder containing the checkpoint
DRIVE_FOLDER_ID = "1eq-Jt6O6gO0Ck_oQYwmmc2qrCVLfKlec"  # <-- update as needed

# ── Binary task (logit_diff metric) ──
# CRC class indices: ADI=0,BACK=1,DEB=2,LYM=3,MUC=4,MUS=5,NORM=6,STR=7,TUM=8
POS_CLASS = 8   # TUM (tumor) — "positive" behavior
NEG_CLASS = 6   # NORM (normal mucosa)

# ── Layers to patch ──
# None = all layers; or specify a subset, e.g. [0, 3, 6, 9, 11]
PATCHING_LAYERS = None

# ── Attribution patching ──
N_SAMPLES_ATTR  = 150   # image pairs for attribution patching scan
N_SAMPLES_EXACT = 30    # image pairs for slow exact patching
N_SAMPLES_SHORT = 100   # image pairs for shortcut detection

# ── Exact patching layers (subset after attr scan) ──
# Override with a short list after inspecting the attr heatmap
EXACT_LAYERS = None  # None = auto-pick top-5 layers from attr scan

# ── Corruption config ──
CORRUPTIONS = [
    "null_baseline",
    "patch_shuffle",
    "gaussian_noise",
    "horizontal_flip",
    "color_jitter",
]

# ── General ──
BATCH_SIZE  = 8
IMAGE_SIZE  = 224
NUM_WORKERS = 2
SEED        = 42

# Paths (Kaggle)
TRAINVAL_ROOT_STR = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
TEST_ROOT_STR     = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K"
PROJECT_ROOT      = "/kaggle/working/xai-vit-medical"

In [ ]:
# ============================================================
# Per-model configuration
# ============================================================

MODEL_CONFIGS = {
    "deit_base": {
        "source"          : "timm",
        "timm_name"       : "deit_base_distilled_patch16_224",
        "checkpoint_fname": "deit_base_best.pth",
        "num_layers"      : 12,
        "num_heads"       : 12,
        "head_dim"        : 64,
        "backbone_prefix" : "",
    },
    "dinov2": {
        "source"          : "torch_hub",
        "timm_name"       : None,
        "checkpoint_fname": "dinov2_best.pth",
        "num_layers"      : 12,
        "num_heads"       : 12,
        "head_dim"        : 64,
        "backbone_prefix" : "backbone.",
    },
}

assert MODEL_NAME in MODEL_CONFIGS, f"Unknown MODEL_NAME: {MODEL_NAME}"
MCFG = MODEL_CONFIGS[MODEL_NAME]

if PATCHING_LAYERS is None:
    PATCHING_LAYERS = list(range(MCFG["num_layers"]))

print(f"Model           : {MODEL_NAME}")
print(f"Checkpoint      : {MCFG['checkpoint_fname']}")
print(f"Binary task     : class {POS_CLASS} vs class {NEG_CLASS}")
print(f"Layers to patch : {PATCHING_LAYERS}")
print(f"Corruptions     : {CORRUPTIONS}")

In [ ]:
import csv
import gc
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader, Subset
from tqdm.notebook import tqdm

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed
from src.xai.mechanistic.activation_patching import (
    logit_diff_metric,
    corrupt_null_baseline,
    corrupt_patch_shuffle,
    corrupt_gaussian_noise,
    corrupt_horizontal_flip,
    corrupt_color_jitter,
    CORRUPTION_REGISTRY,
    activation_patching,
    patching_scan,
    attribution_patching,
    attribution_patching_heads,
    detect_shortcuts,
)

set_seed(SEED)
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = list(DEFAULT_CRC_CLASSES)
NUM_CLASSES = len(CLASS_NAMES)

SAVE_DIR = Path(f"{PROJECT_ROOT}/outputs/patching/{MODEL_NAME}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRAINVAL_ROOT = Path(TRAINVAL_ROOT_STR)
TEST_ROOT     = Path(TEST_ROOT_STR)

METRIC = logit_diff_metric(positive_class=POS_CLASS, negative_class=NEG_CLASS)

print(f"Device : {DEVICE}")
print(f"Metric : logit[{CLASS_NAMES[POS_CLASS]}] - logit[{CLASS_NAMES[NEG_CLASS]}]")
print(f"Output : {SAVE_DIR}")

In [ ]:
# ── Download checkpoint from Google Drive ──────────────────────────────────
CHECKPOINT_FNAME = MCFG["checkpoint_fname"]
CKPT_LOCAL = Path(f"{PROJECT_ROOT}/outputs/models/{CHECKPOINT_FNAME}")
CKPT_LOCAL.parent.mkdir(parents=True, exist_ok=True)

if not CKPT_LOCAL.exists():
    file_list = drive.ListFile(
        {"q": f"'{DRIVE_FOLDER_ID}' in parents and title='{CHECKPOINT_FNAME}' and trashed=false"}
    ).GetList()
    if not file_list:
        raise FileNotFoundError(
            f"{CHECKPOINT_FNAME} not found in Drive folder {DRIVE_FOLDER_ID}."
        )
    file_list[0].GetContentFile(str(CKPT_LOCAL))
    print(f"Downloaded: {CHECKPOINT_FNAME}")
else:
    print(f"Checkpoint present: {CKPT_LOCAL}")

In [ ]:
# ── Build model ────────────────────────────────────────────────────────────

def load_state(model: nn.Module, ckpt_path: Path) -> dict:
    state = torch.load(ckpt_path, map_location="cpu")
    if "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
        return {k: v for k, v in state.items() if k != "model_state_dict"}
    cleaned = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(cleaned, strict=False)
    return {}


if MCFG["source"] == "torch_hub":
    from src.models.dinov2 import DINOv2Classifier
    model = DINOv2Classifier(num_classes=NUM_CLASSES, usage_mode="frozen_linear_probe")
else:
    model = timm.create_model(MCFG["timm_name"], pretrained=False, num_classes=NUM_CLASSES)

meta = load_state(model, CKPT_LOCAL)
model = model.to(DEVICE).eval()
BACKBONE_PREFIX = MCFG["backbone_prefix"]

print(f"Model       : {MODEL_NAME}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")
print(f"Epoch       : {meta.get('epoch', '?')}")
print(f"Val acc     : {meta.get('val_acc', '?')}")

In [ ]:
# ── Dataset — balanced TUM / NORM subset ───────────────────────────────────
# Activation patching works best on a binary discrimination task.
# We use images from POS_CLASS and NEG_CLASS only.

crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=0.25,
    random_state=SEED,
)

test_dataset = CRCHistologyDataset(
    split="test",
    splits=crc_splits,
    image_size=IMAGE_SIZE,
    return_id=False,
)

# Filter to POS_CLASS and NEG_CLASS only, balanced
N_PER_CLASS = max(N_SAMPLES_ATTR, N_SAMPLES_SHORT) // 2 + 20
counts: dict[int, int] = {POS_CLASS: 0, NEG_CLASS: 0}
binary_indices: list[int] = []
for idx, lbl in enumerate(test_dataset.labels):
    lbl = int(lbl)
    if lbl in counts and counts[lbl] < N_PER_CLASS:
        binary_indices.append(idx)
        counts[lbl] += 1

binary_dataset = Subset(test_dataset, binary_indices)
binary_loader  = DataLoader(
    binary_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

print(f"Binary subset: {CLASS_NAMES[POS_CLASS]}={counts[POS_CLASS]}  "
      f"{CLASS_NAMES[NEG_CLASS]}={counts[NEG_CLASS]}  total={len(binary_indices)}")

# Quick model sanity check on binary task
@torch.no_grad()
def binary_accuracy(loader, max_batches=5):
    correct = total = 0
    for i, (imgs, lbls) in enumerate(loader):
        if i >= max_batches: break
        imgs = imgs.to(DEVICE)
        out  = model(imgs)
        if hasattr(out, "logits"): out = out.logits
        if isinstance(out, tuple): out = out[0]
        preds = out.argmax(1).cpu()
        # Only count POS/NEG class predictions
        for p, l in zip(preds.tolist(), lbls.tolist()):
            if l in (POS_CLASS, NEG_CLASS):
                correct += int(p == l)
                total   += 1
    return correct / max(total, 1)

acc = binary_accuracy(binary_loader)
print(f"Binary accuracy ({CLASS_NAMES[POS_CLASS]} vs {CLASS_NAMES[NEG_CLASS]}): {acc:.3f}")

In [ ]:
# ── Preview corruptions on a sample image ──────────────────────────────────
sample_imgs, sample_lbls = next(iter(binary_loader))
sample_img = sample_imgs[0]  # (C, H, W)

def denormalize(img_chw: torch.Tensor) -> np.ndarray:
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img  = img_chw.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(img * std + mean, 0, 1)

corrupted_samples = {
    "original"       : sample_img,
    "null_baseline"  : corrupt_null_baseline(sample_img),
    "patch_shuffle"  : corrupt_patch_shuffle(sample_img, ratio=0.3),
    "gaussian_noise" : corrupt_gaussian_noise(sample_img, std=0.3),
    "horizontal_flip": corrupt_horizontal_flip(sample_img),
    "color_jitter"   : corrupt_color_jitter(sample_img),
}

n = len(corrupted_samples)
fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
for ax, (name, img) in zip(axes, corrupted_samples.items()):
    ax.imshow(denormalize(img))
    ax.set_title(name, fontsize=8)
    ax.axis("off")
lbl_name = CLASS_NAMES[int(sample_lbls[0])]
plt.suptitle(f"Corruption preview — GT: {lbl_name}", fontsize=11)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / "corruption_preview.png"), dpi=150)
plt.show()
print(f"Saved: {SAVE_DIR / 'corruption_preview.png'}")

In [ ]:
# ── Attribution Patching — fast scan (all layers × components) ─────────────
# For each corruption, accumulate attribution scores across N_SAMPLES_ATTR images.
# This identifies which (layer, component) pairs matter most for TUM vs NORM.

COMPONENTS = ["attn_output", "mlp_output", "resid_post"]

# Accumulate per-corruption IE estimates: dict[corruption → list[dict[(layer,comp) → score]]]
attr_by_corruption: dict[str, list[dict]] = {c: [] for c in CORRUPTIONS}

n_processed = 0
for batch in tqdm(binary_loader, desc="Attribution patching scan"):
    if n_processed >= N_SAMPLES_ATTR:
        break
    imgs, lbls = batch
    imgs = imgs.to(DEVICE)

    for i in range(imgs.shape[0]):
        if n_processed >= N_SAMPLES_ATTR:
            break
        clean_img = imgs[i : i + 1]

        for corr_name in CORRUPTIONS:
            corr_fn = CORRUPTION_REGISTRY.get(corr_name)
            if corr_fn is None:
                continue
            try:
                corrupted = corr_fn(clean_img[0]).unsqueeze(0).to(DEVICE)
            except Exception:
                continue

            try:
                scores = attribution_patching(
                    model=model,
                    clean_input=clean_img,
                    corrupt_input=corrupted,
                    layers=PATCHING_LAYERS,
                    components=COMPONENTS,
                    metric=METRIC,
                    backbone_prefix=BACKBONE_PREFIX,
                )
                attr_by_corruption[corr_name].append(scores)
            except Exception as e:
                print(f"  [warn] attr patching failed ({corr_name}): {e}")
        n_processed += 1

print(f"\nProcessed {n_processed} images")
for c, samples in attr_by_corruption.items():
    print(f"  {c}: {len(samples)} valid samples")

In [ ]:
# ── Attribution Patching — IE heatmaps ────────────────────────────────────

def build_ie_matrix(
    sample_list: list[dict],
    layers: list[int],
    components: list[str],
) -> tuple[np.ndarray, np.ndarray]:
    """Aggregate IE scores → mean/std matrix (n_layers, n_components)."""
    arr = np.zeros((len(sample_list), len(layers), len(components)))
    for si, sample in enumerate(sample_list):
        for li, layer in enumerate(layers):
            for ci, comp in enumerate(components):
                arr[si, li, ci] = sample.get((layer, comp), 0.0)
    return arr.mean(0), arr.std(0)


n_corr = len(CORRUPTIONS)
fig, axes = plt.subplots(1, n_corr, figsize=(4.5 * n_corr, 5), squeeze=False)
fig.suptitle(f"{MODEL_NAME} — Attribution Patching IE heatmaps\n"
             f"({CLASS_NAMES[POS_CLASS]} vs {CLASS_NAMES[NEG_CLASS]})", fontsize=12)

ie_matrices: dict[str, np.ndarray] = {}  # mean IE per corruption

for col_i, corr_name in enumerate(CORRUPTIONS):
    samples = attr_by_corruption[corr_name]
    if not samples:
        axes[0, col_i].set_visible(False)
        continue

    mean_ie, std_ie = build_ie_matrix(samples, PATCHING_LAYERS, COMPONENTS)
    ie_matrices[corr_name] = mean_ie

    ax = axes[0, col_i]
    vmax = max(abs(mean_ie).max(), 1e-6)
    im = ax.imshow(mean_ie, aspect="auto", cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax, origin="lower")
    ax.set_xticks(range(len(COMPONENTS)))
    ax.set_xticklabels([c.replace("_output", "").replace("resid_", "resid\n") for c in COMPONENTS],
                       fontsize=8)
    ax.set_yticks(range(len(PATCHING_LAYERS)))
    ax.set_yticklabels(PATCHING_LAYERS, fontsize=7)
    ax.set_xlabel("Component")
    ax.set_ylabel("Layer")
    ax.set_title(corr_name, fontsize=9)
    plt.colorbar(im, ax=ax, label="Mean IE")

plt.tight_layout()
plt.savefig(str(SAVE_DIR / "attr_patching_heatmaps.png"), dpi=150, bbox_inches="tight")
plt.show()

# Save raw IE matrices
np.save(str(SAVE_DIR / "attr_ie_matrices.npy"),
        {c: ie_matrices[c] for c in ie_matrices})
print(f"\nSaved IE heatmaps and raw matrices → {SAVE_DIR}")

In [ ]:
# ── Identify top components from attribution scan ──────────────────────────
# Average across all corruptions to find universally important components.

if ie_matrices:
    mean_across_corr = np.mean(list(ie_matrices.values()), axis=0)  # (n_layers, n_comp)
    abs_mean = np.abs(mean_across_corr)

    # Top-5 (layer, component) pairs by absolute mean IE
    flat_idx = np.argsort(abs_mean.flatten())[::-1][:10]
    top_pairs = []
    print(f"Top 10 (layer, component) pairs by mean |IE| across all corruptions:")
    for rank, fi in enumerate(flat_idx):
        li, ci = divmod(fi, len(COMPONENTS))
        layer_idx  = PATCHING_LAYERS[li]
        comp_name  = COMPONENTS[ci]
        ie_val     = mean_across_corr[li, ci]
        top_pairs.append((layer_idx, comp_name))
        print(f"  #{rank+1:2d}  L{layer_idx:2d} / {comp_name:15s}  mean IE = {ie_val:+.4f}")

    # Auto-select layers for exact patching
    if EXACT_LAYERS is None:
        layer_importance = abs_mean.max(axis=1)  # max over components per layer
        top_layer_indices = np.argsort(layer_importance)[::-1][:5]
        EXACT_LAYERS = sorted([PATCHING_LAYERS[i] for i in top_layer_indices])
        print(f"\nAuto-selected layers for exact patching: {EXACT_LAYERS}")
else:
    print("No IE matrices available — using default exact layers")
    if EXACT_LAYERS is None:
        EXACT_LAYERS = PATCHING_LAYERS[-4:]  # last 4 layers
    top_pairs = [(l, c) for l in EXACT_LAYERS for c in COMPONENTS]

In [ ]:
# ── Exact Activation Patching — denoising + noising ───────────────────────
# Run exact patching on top layers for the most informative corruption.
# We use null_baseline (maximal perturbation) for the main analysis.

EXACT_CORRUPTION = "null_baseline"  # most informative for denoising
EXACT_COMPONENTS = ["attn_output", "mlp_output"]

exact_results: dict[str, dict] = {"denoising": {}, "noising": {}}
# Format: {direction: {(layer, comp): result_dict}}

n_processed = 0
denoise_acc: dict[tuple, list[float]] = defaultdict(list)
noising_acc : dict[tuple, list[float]] = defaultdict(list)

corr_fn = CORRUPTION_REGISTRY[EXACT_CORRUPTION]

for batch in tqdm(binary_loader, desc="Exact patching"):
    if n_processed >= N_SAMPLES_EXACT:
        break
    imgs, lbls = batch
    imgs = imgs.to(DEVICE)

    for i in range(imgs.shape[0]):
        if n_processed >= N_SAMPLES_EXACT:
            break
        clean_img = imgs[i : i + 1]
        corrupted = corr_fn(clean_img[0]).unsqueeze(0).to(DEVICE)

        for direction, acc_dict in [("denoising", denoise_acc), ("noising", noising_acc)]:
            for layer_idx in EXACT_LAYERS:
                for component in EXACT_COMPONENTS:
                    try:
                        res = activation_patching(
                            model=model,
                            clean_input=clean_img,
                            corrupt_input=corrupted,
                            target_layer=layer_idx,
                            target_component=component,
                            direction=direction,
                            metric=METRIC,
                            backbone_prefix=BACKBONE_PREFIX,
                        )
                        acc_dict[(layer_idx, component)].append(res["indirect_effect"])
                    except Exception as e:
                        pass
        n_processed += 1

print(f"Exact patching done — {n_processed} images")

# Aggregate
for direction, acc_dict in [("denoising", denoise_acc), ("noising", noising_acc)]:
    exact_results[direction] = {
        k: {"mean_ie": float(np.mean(v)), "std_ie": float(np.std(v)), "n": len(v)}
        for k, v in acc_dict.items()
    }

In [ ]:
# ── Exact patching — comparison heatmap (denoising vs noising) ────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"{MODEL_NAME} — Exact Activation Patching\n"
             f"Corruption: {EXACT_CORRUPTION}  |  Metric: logit_diff "
             f"({CLASS_NAMES[POS_CLASS]} vs {CLASS_NAMES[NEG_CLASS]})", fontsize=11)

for ax, (direction, res_dict) in zip(axes, exact_results.items()):
    if not res_dict:
        ax.set_visible(False)
        continue

    n_layers = len(EXACT_LAYERS)
    n_comp   = len(EXACT_COMPONENTS)
    mat = np.zeros((n_layers, n_comp))
    for li, layer in enumerate(EXACT_LAYERS):
        for ci, comp in enumerate(EXACT_COMPONENTS):
            key = (layer, comp)
            if key in res_dict:
                mat[li, ci] = res_dict[key]["mean_ie"]

    vmax = max(abs(mat).max(), 1e-6)
    im = ax.imshow(mat, aspect="auto", cmap="RdYlGn", vmin=-vmax, vmax=vmax, origin="lower")

    # Annotate cells
    for li in range(n_layers):
        for ci in range(n_comp):
            ax.text(ci, li, f"{mat[li, ci]:.2f}",
                    ha="center", va="center", fontsize=9,
                    color="white" if abs(mat[li, ci]) > vmax * 0.5 else "black")

    ax.set_xticks(range(n_comp))
    ax.set_xticklabels([c.replace("_output", "") for c in EXACT_COMPONENTS])
    ax.set_yticks(range(n_layers))
    ax.set_yticklabels([f"L{l}" for l in EXACT_LAYERS])
    ax.set_xlabel("Component")
    ax.set_ylabel("Layer")
    ax.set_title(f"{direction} IE  (1.0 = full "
                 + ("recovery" if direction == "denoising" else "disruption") + ")")
    plt.colorbar(im, ax=ax, label="Mean IE")

plt.tight_layout()
plt.savefig(str(SAVE_DIR / "exact_patching_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

# Print top components
print("\nTop denoising components (most SUFFICIENT):")
dn = {k: v["mean_ie"] for k, v in exact_results["denoising"].items()}
for k, ie in sorted(dn.items(), key=lambda x: abs(x[1]), reverse=True)[:5]:
    print(f"  L{k[0]:2d} / {k[1]:15s}  IE = {ie:+.4f}")

print("\nTop noising components (most NECESSARY):")
ni = {k: v["mean_ie"] for k, v in exact_results["noising"].items()}
for k, ie in sorted(ni.items(), key=lambda x: abs(x[1]), reverse=True)[:5]:
    print(f"  L{k[0]:2d} / {k[1]:15s}  IE = {ie:+.4f}")

In [ ]:
# ── Per-head attribution patching ─────────────────────────────────────────
# Which attention heads are most causally important for TUM vs NORM?

NUM_HEADS = MCFG["num_heads"]
HEAD_DIM  = MCFG["head_dim"]

# Scan all layers with a subset of images (head patching is cheaper than exact but slower than attr)
N_HEAD_SAMPLES = min(50, N_SAMPLES_ATTR)
head_scores_list: list[dict] = []

n_processed = 0
for batch in tqdm(binary_loader, desc="Head attribution patching"):
    if n_processed >= N_HEAD_SAMPLES:
        break
    imgs, lbls = batch
    imgs = imgs.to(DEVICE)

    for i in range(imgs.shape[0]):
        if n_processed >= N_HEAD_SAMPLES:
            break
        clean_img = imgs[i : i + 1]
        corrupted = corrupt_null_baseline(clean_img[0]).unsqueeze(0).to(DEVICE)

        try:
            head_scores = attribution_patching_heads(
                model=model,
                clean_input=clean_img,
                corrupt_input=corrupted,
                layers=PATCHING_LAYERS,
                num_heads=NUM_HEADS,
                head_dim=HEAD_DIM,
                metric=METRIC,
                backbone_prefix=BACKBONE_PREFIX,
            )
            head_scores_list.append(head_scores)
        except Exception as e:
            print(f"  [warn] head patching failed: {e}")
        n_processed += 1

print(f"Head attribution: {n_processed} samples processed")

In [ ]:
# ── Per-head IE heatmap (layer × head) ────────────────────────────────────

if head_scores_list:
    n_layers = len(PATCHING_LAYERS)
    head_mat = np.zeros((n_layers, NUM_HEADS))

    for sample in head_scores_list:
        for li, layer in enumerate(PATCHING_LAYERS):
            for h in range(NUM_HEADS):
                head_mat[li, h] += sample.get((layer, h), 0.0)
    head_mat /= len(head_scores_list)

    fig, ax = plt.subplots(figsize=(min(NUM_HEADS * 0.8, 14), 6))
    vmax = max(abs(head_mat).max(), 1e-6)
    im = ax.imshow(head_mat, aspect="auto", cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax, origin="lower")
    ax.set_xticks(range(NUM_HEADS))
    ax.set_xticklabels([f"H{h}" for h in range(NUM_HEADS)], fontsize=8)
    ax.set_yticks(range(n_layers))
    ax.set_yticklabels([f"L{l}" for l in PATCHING_LAYERS], fontsize=8)
    ax.set_xlabel("Attention head")
    ax.set_ylabel("Layer")
    ax.set_title(
        f"{MODEL_NAME} — Head attribution patching IE\n"
        f"({CLASS_NAMES[POS_CLASS]} vs {CLASS_NAMES[NEG_CLASS]}, "
        f"null_baseline corruption, n={len(head_scores_list)})"
    )
    plt.colorbar(im, ax=ax, label="Mean attribution IE")
    plt.tight_layout()
    plt.savefig(str(SAVE_DIR / "head_attribution_heatmap.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # Top-5 heads
    flat_idx = np.argsort(np.abs(head_mat).flatten())[::-1][:10]
    print("Top 10 attention heads by |mean IE|:")
    for rank, fi in enumerate(flat_idx):
        li, h = divmod(fi, NUM_HEADS)
        print(f"  #{rank+1:2d}  L{PATCHING_LAYERS[li]:2d} H{h:2d}  IE = {head_mat[li, h]:+.4f}")

    np.save(str(SAVE_DIR / "head_attribution_matrix.npy"), head_mat)
else:
    print("No head attribution results available.")

In [ ]:
# ── Shortcut Detection ─────────────────────────────────────────────────────
# Which components show high IE specifically on semantic corruptions
# (color_jitter, horizontal_flip) that should NOT matter for CRC classification?
# High IE on these → potential shortcut reliance.

SHORTCUT_CORRUPTIONS = ["color_jitter", "horizontal_flip", "patch_shuffle"]
IE_THRESHOLD = 0.15  # lower than default 0.3 for sensitivity

shortcut_results = detect_shortcuts(
    model=model,
    dataloader=binary_loader,
    layers=PATCHING_LAYERS,
    components=COMPONENTS,
    corruptions=SHORTCUT_CORRUPTIONS,
    metric=METRIC,
    max_samples=N_SAMPLES_SHORT,
    ie_threshold=IE_THRESHOLD,
    backbone_prefix=BACKBONE_PREFIX,
    device=DEVICE,
)

print("\nShortcut detection summary:")
all_shortcuts = []
for corr_name, res in shortcut_results.items():
    n_flags = len(res.get("shortcuts", []))
    print(f"  {corr_name}: {res.get('n_samples', 0)} samples — {n_flags} flagged components")
    for s in res.get("shortcuts", []):
        print(f"    L{s['layer']:2d} / {s['component']:15s}  IE = {s['mean_ie']:+.4f}")
        all_shortcuts.append({"corruption": corr_name, **s})

if not all_shortcuts:
    print(f"  → No shortcuts detected at IE threshold {IE_THRESHOLD}")
else:
    print(f"\n  ⚠  {len(all_shortcuts)} shortcut flags total")

with open(SAVE_DIR / "shortcut_detection.json", "w") as f:
    json.dump(
        {c: {"n_samples": r["n_samples"], "shortcuts": r["shortcuts"]}
         for c, r in shortcut_results.items()},
        f, indent=2
    )

In [ ]:
# ── Shortcut heatmaps ─────────────────────────────────────────────────────

n_sc = len(shortcut_results)
if n_sc > 0:
    fig, axes = plt.subplots(1, n_sc, figsize=(5 * n_sc, 6), squeeze=False)
    fig.suptitle(
        f"{MODEL_NAME} — Shortcut Detection\n"
        f"High IE on semantic corruptions = potential shortcut  (threshold={IE_THRESHOLD})",
        fontsize=11
    )

    for col_i, (corr_name, res) in enumerate(shortcut_results.items()):
        ax = axes[0, col_i]
        mean_ie = res["mean_ie_matrix"].numpy()  # (n_layers, n_comp)

        vmax = max(abs(mean_ie).max(), 1e-6)
        im = ax.imshow(mean_ie, aspect="auto", cmap="OrRd",
                       vmin=0, vmax=vmax, origin="lower")

        # Mark flagged cells
        for li, layer in enumerate(res["layers"]):
            for ci, comp in enumerate(res["components"]):
                val = mean_ie[li, ci]
                if abs(val) > IE_THRESHOLD:
                    ax.add_patch(plt.Rectangle(
                        (ci - 0.5, li - 0.5), 1, 1,
                        fill=False, edgecolor="red", linewidth=2
                    ))
                ax.text(ci, li, f"{val:.2f}", ha="center", va="center",
                        fontsize=6, color="white" if val > vmax * 0.6 else "black")

        ax.set_xticks(range(len(COMPONENTS)))
        ax.set_xticklabels([c.replace("_output","").replace("resid_","r\n") for c in COMPONENTS],
                           fontsize=8)
        ax.set_yticks(range(len(PATCHING_LAYERS)))
        ax.set_yticklabels(PATCHING_LAYERS, fontsize=7)
        ax.set_xlabel("Component")
        ax.set_ylabel("Layer")
        ax.set_title(f"{corr_name}\n(n={res['n_samples']})", fontsize=9)
        plt.colorbar(im, ax=ax, label="Mean |IE|")

    plt.tight_layout()
    plt.savefig(str(SAVE_DIR / "shortcut_heatmaps.png"), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Layer profile: IE by depth (circuit localization) ─────────────────────
# For each corruption, plot mean IE summed over components per layer.
# This shows at which depth the model encodes the relevant features.

fig, ax = plt.subplots(figsize=(10, 5))

for corr_name, samples in attr_by_corruption.items():
    if not samples:
        continue
    mean_ie, _ = build_ie_matrix(samples, PATCHING_LAYERS, COMPONENTS)
    layer_profile = np.abs(mean_ie).mean(axis=1)  # mean over components
    ax.plot(PATCHING_LAYERS, layer_profile, marker="o", label=corr_name)

ax.set_xlabel("Layer")
ax.set_ylabel("Mean |IE| (averaged over components)")
ax.set_title(
    f"{MODEL_NAME} — IE by layer depth\n"
    f"Task: {CLASS_NAMES[POS_CLASS]} vs {CLASS_NAMES[NEG_CLASS]}"
)
ax.legend()
ax.grid(True)
ax.set_xticks(PATCHING_LAYERS)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / "layer_profile.png"), dpi=150)
plt.show()

In [ ]:
# ── Save all results ───────────────────────────────────────────────────────

# Exact patching results (serialisable)
exact_serialisable = {}
for direction, res_dict in exact_results.items():
    exact_serialisable[direction] = {
        f"{k[0]}_{k[1]}": v for k, v in res_dict.items()
    }

# Attribution patching summary per corruption
attr_summary = {}
for corr_name, samples in attr_by_corruption.items():
    if not samples:
        continue
    mean_ie, std_ie = build_ie_matrix(samples, PATCHING_LAYERS, COMPONENTS)
    top_idx = int(np.argmax(np.abs(mean_ie)))
    li, ci  = divmod(top_idx, len(COMPONENTS))
    attr_summary[corr_name] = {
        "n_samples"       : len(samples),
        "max_abs_ie"      : float(np.abs(mean_ie).max()),
        "top_layer"       : PATCHING_LAYERS[li],
        "top_component"   : COMPONENTS[ci],
        "top_mean_ie"     : float(mean_ie[li, ci]),
    }

global_summary = {
    "model"            : MODEL_NAME,
    "pos_class"        : CLASS_NAMES[POS_CLASS],
    "neg_class"        : CLASS_NAMES[NEG_CLASS],
    "patching_layers"  : PATCHING_LAYERS,
    "components"       : COMPONENTS,
    "corruptions"      : CORRUPTIONS,
    "exact_corruption" : EXACT_CORRUPTION,
    "exact_layers"     : EXACT_LAYERS,
    "attr_summary"     : attr_summary,
    "exact_patching"   : exact_serialisable,
    "n_shortcuts"      : len(all_shortcuts),
}

with open(SAVE_DIR / "patching_summary.json", "w") as f:
    json.dump(global_summary, f, indent=2)

print(f"Saved: {SAVE_DIR / 'patching_summary.json'}")

# Attribution score CSV
with open(SAVE_DIR / "attr_scores.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["corruption", "layer", "component", "mean_ie", "std_ie"])
    for corr_name, samples in attr_by_corruption.items():
        if not samples:
            continue
        mean_ie, std_ie = build_ie_matrix(samples, PATCHING_LAYERS, COMPONENTS)
        for li, layer in enumerate(PATCHING_LAYERS):
            for ci, comp in enumerate(COMPONENTS):
                w.writerow([corr_name, layer, comp,
                            float(mean_ie[li, ci]), float(std_ie[li, ci])])

print(f"Saved: {SAVE_DIR / 'attr_scores.csv'}")

In [ ]:
# ── Upload results to Google Drive ─────────────────────────────────────────
RESULTS_DRIVE_FOLDER = DRIVE_FOLDER_ID

files_to_upload = [
    SAVE_DIR / "patching_summary.json",
    SAVE_DIR / "attr_scores.csv",
    SAVE_DIR / "shortcut_detection.json",
    SAVE_DIR / "corruption_preview.png",
    SAVE_DIR / "attr_patching_heatmaps.png",
    SAVE_DIR / "exact_patching_heatmap.png",
    SAVE_DIR / "head_attribution_heatmap.png",
    SAVE_DIR / "shortcut_heatmaps.png",
    SAVE_DIR / "layer_profile.png",
]

for fpath in files_to_upload:
    fpath = Path(fpath)
    if not fpath.exists():
        print(f"  Not found (skipped): {fpath.name}")
        continue
    drive_file = drive.CreateFile({
        "title"  : f"{MODEL_NAME}_patching_{fpath.name}",
        "parents": [{"id": RESULTS_DRIVE_FOLDER}],
    })
    drive_file.SetContentFile(str(fpath))
    drive_file.Upload()
    print(f"  Uploaded: {MODEL_NAME}_patching_{fpath.name}")

print("\nDone.")

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup done.")